# Monads

Navigating through Functors and Monads

## Functors

A functor is simply a trait (or interface) that implements the map method.

 - Functors always have a **map** function
 - Functors can always be **composed**


The signature of a `Functor` is as follows, featureing the `map` method:

trait Functor[F[_]] : 
	def map[A, B](fa: F[A])(f: A => B): F[B] 

In [1]:
import $ivy.`org.typelevel::cats-core:2.13.0`

Downloaded https://repo1.maven.org/maven2/org/typelevel/cats-core_3/2.13.0/cats-core_3-2.13.0.pom
Downloaded https://repo1.maven.org/maven2/org/typelevel/cats-kernel_3/2.13.0/cats-kernel_3-2.13.0.pom


import $ivy.$                                


Using a functr with Cats essentially allows using the map function as one would expect. 

For example on an `Option`:

In [ ]:
import cats.Functor

Functor[Option].map(Some(2))(_ + 1)

In [ ]:
Functor[Option].map(None:Option[Int])(_ + 1)

Then we can do compositions of Functors for different types:

In [ ]:
val funcOpt = Functor[Option]
val funcList = Functor[List]
val functCompose = (funcList compose funcOpt)

Once we have this new composed `Functor`, we can apply it to other instances:

In [ ]:
val list = List(3,6,4,2).map(Option(_))++List(None)

functCompose.map(list)(_ * 3)

As expected, we can define map for a `Functor` on custom classes:

In [ ]:
case class Evaluation[A](grade:A,remark:String)

In [ ]:
given Functor[Evaluation] with
  def map[A,B](fa: Evaluation[A])(f: A => B) =
     Evaluation(f(fa.grade),fa.remark)

Then we can use it on concrete instances, with a bit of help from Cats syntax helpers:

In [ ]:
val e1 = Evaluation(3.5,"not good")

import cats.syntax.functor._
e1.map(_.toInt)

Some helpers exist like `fproduct` to apply map in a Functor way:

In [ ]:
val count_a = (word:String) => word.count(_ == 'a')
val source = List("Avatar", "the", "last","airbender")
val product = Functor[List].fproduct(source)(count_a).toMap

product.get("Avatar").getOrElse(0) 

## Monads 

A **monad** is a trait (or interface) that implements two methods: 

- **flatMap** and **unit**

Every monad is also a *functor*: must have a **map** function. 

We can have *map* combining *flatmap* and *unit*.


The Cats signature includes obviously the `flatmap` method, and `pure`, which represents the *unit*:

```
trait Monad[F[_]] : 

	def pure[A](a: A): F[A] 

	def flatMap[A, B](fa: F[A])(f: A => F[B]): F[B]

```

Now to use it in Cats, it is very similar to a `Functor`. We can apply it to `Option`:

In [ ]:
import cats.Monad

val anOpt = Monad[Option].pure(5)

In [ ]:
val otherOpt = Monad[Option].flatMap(anOpt)(e => Option(e + 2))

In [ ]:
val anotherOpt = Monad[Option].map(otherOpt)(e => 300 + e)

Obviously goes as well perfect with collections: 

In [ ]:
import cats.Monad

val aList= Monad[List].pure(2)

val otherList = 
  Monad[List].flatMap(List(4,6,2))(e => List(e-1,e,e+1))

Now we can, as usual use some Cats syntax helpers to make it easier:

In [ ]:
Monad[Option].pure(25)

In [ ]:
import cats.syntax.applicative._
25.pure[Option]

Using `Monad` now for custom classes:

In [ ]:
enum Unit(val code:String) :
  case Kelvin extends Unit("K")
  case Celsius extends Unit("C")
  case Farenheit extends Unit("F")


In [ ]:
case class Temperature[T](value:T,unit:Unit=Unit.Celsius):
  override def toString() = s"${value.toString}°${unit.code}"


We can use the `Monad` as usual with Cats with a typeclass:

Notice that we also need to implement the `tailRecM` but as we don't use it we leave it unimplemented.

In [ ]:
given Monad[Temperature] with
  def flatMap[A,B](fa:Temperature[A])(f:A => Temperature[B]):Temperature[B] =
     f(fa.value)

  def pure[A](x: A): Temperature[A] = Temperature(x)

  def tailRecM[A,B](a: A)(f: A => Temperature[Either[A, B]]):Temperature[B] = 
     ???

Now we can simply use the Monad:

In [ ]:
Monad[Temperature].pure(34)

Or easier:

In [ ]:
34.pure[Temperature]

Beyond `pure` which only envelops a value into the monad, we can make use of flatmap to do some operations:

In [ ]:
val temp = Temperature(34)

Monad[Temperature].flatMap(temp)(t => Temperature(t+273.15,Unit.Kelvin))

Or use some more Cats syntax coating to make it easier:

In [ ]:
import cats.syntax.monad._
import cats.syntax.flatMap._
import cats.syntax.functor._

temp.flatMap(e => Temperature(e+273.15,Unit.Kelvin))

Then composing monads (following the Functor way) becomes totally possible:

In [ ]:
val optTemp = Monad[Option] compose Monad[Temperature]

optTemp.pure(None)

In [ ]:
optTemp.pure(45)

## Either

One classical Monad is `Either` representing a value with two possible types:

In [2]:
val either1:Either[String,Int] = Left("4")

val either2:Either[String,Int] = Right(4)


either1: Either[String, Int] = Left(value = "4")
either2: Either[String, Int] = Right(value = 4)

Typical use if to get right or wrong values:

In [3]:
import cats.syntax.either._

def computation[T](param:T):Either[String,Double] = param match
  case t:Int => (t*0.5).asRight
  case d:Double => (d*0.5).asRight
  case _ => "Not a good param".asLeft

import cats.syntax.either._


defined function computation

In [4]:
computation("10")

res4: Either[String, Double] = Left(value = "Not a good param")

With the help of Cats we can get values to be enveloped in Either:

In [5]:
import cats.syntax.either._

3.asLeft

import cats.syntax.either._


res5_1: Either[Int, Nothing] = Left(value = 3)

And as we are in the realm of Monads, we can be naturally led to use flatMap:

In [6]:
import cats.syntax.functor._
import cats.syntax.monad._
import cats.syntax.flatMap._

val f = computation(10).flatMap(a => a.asRight)

import cats.syntax.functor._

import cats.syntax.monad._

import cats.syntax.flatMap._


f: Either[String, Double] = Right(value = 5.0)

Either has other functionality, like the ensure mehtod for validation:

In [7]:
val positive = (i:Int) => i.asRight[String].ensure("Should be positive")(_ > 0)

positive(-3)

positive(2)

positive: Int => Either[String, Int] = ammonite.$sess.cmd7$Helper$$Lambda/0x00007f6af0b4ecb0@5904f2d8
res7_1: Either[String, Int] = Left(value = "Should be positive")
res7_2: Either[String, Int] = Right(value = 2)

Or utilities to swap form Left to Right:

In [8]:
123.asRight[String].swap


res8: Either[Int, String] = Left(value = 123)

Another goodie is the `catchOnly` to deal with concrete Exceptions:

In [9]:
Either.catchOnly[NumberFormatException]("123a".toInt)

res9: Either[NumberFormatException, Int] = Left(
  value = java.lang.NumberFormatException: For input string: "123a"
)

In [10]:
Either.catchOnly[NumberFormatException](3/0)

java.lang.ArithmeticException: / by zero

## Foldable

Another useful Monad is Foldable, which can be handy to deal with collections to which different variants of `fold` are applicable:

In [17]:
import cats.Foldable
import cats.instances.list
import cats.instances.seq
import cats.syntax.option._

val list:List[Int] = (1 to 10).toList

Foldable[List].foldLeft(list,0)(_ - _)

import cats.Foldable

import cats.instances.list

import cats.instances.seq

import cats.syntax.option._


list: List[Int] = List(1, 2, 3, 4, 5, 6, 7, 8, 9, 10)
res17_5: Int = -55

In [18]:
Foldable[List].fold((1 to 10).toList)

res18: Int = 55

Other variants include foldMap, or forall:

In [19]:
import cats.Foldable
import cats.instances.list
import cats.instances.seq
import cats.syntax.option._

val list = (1 to 10).toList

Foldable[List].foldMap(list)(_.toString)

import cats.Foldable

import cats.instances.list

import cats.instances.seq

import cats.syntax.option._


list: List[Int] = List(1, 2, 3, 4, 5, 6, 7, 8, 9, 10)
res19_5: String = "12345678910"

In [20]:
Foldable[List].forall((1 to 10).toList)(_ <= 3)

res20: Boolean = false

As we are into the world of Functors and monads, we can combine and compose using Foldable:

In [22]:
import cats.syntax.foldable._

(1 to 10).toList.combineAll

import cats.syntax.foldable._


res22_1: Int = 55

In [23]:
(1 to 10).toList.foldMap(_.toString)

res23: String = "12345678910"

In [24]:
val comp = (Foldable[List] compose Foldable[Option])

comp: Foldable[cats.Foldable[[α >: scala.Nothing <: scala.Any] => scala.collection.immutable.List[scala.Option[α]]]] = cats.Foldable$$anon$1@7401292a

In [25]:
comp.combineAll(List(4.some,5.some))

res25: Int = 9